In [1]:
import os
import numpy as np
import cv2
from glob import glob
import matplotlib.pyplot as plt

# Input directory
input_dir = r'C:/Users/athet/Downloads/archive/kaggle_3m'

# Create output directory for saving results (absolute path)
output_dir = os.path.abspath('segmentation_results')
os.makedirs(output_dir, exist_ok=True)

print(f"Input directory: {input_dir}")
print(f"Output directory (absolute): {output_dir}")
print("Trying to list a few entries from the input directory to confirm path is correct:")
try:
    entries = os.listdir(input_dir)
    print(entries[:10])
except Exception as e:
    print("Could not list input_dir contents:", e)


Input directory: C:/Users/athet/Downloads/archive/kaggle_3m
Output directory (absolute): c:\Users\athet\OneDrive\Documents\ULL Admission\machine learning\FInal Project Models\segmentation_results
Trying to list a few entries from the input directory to confirm path is correct:
['data.csv', 'README.md', 'TCGA_CS_4941_19960909', 'TCGA_CS_4942_19970222', 'TCGA_CS_4943_20000902', 'TCGA_CS_4944_20010208', 'TCGA_CS_5393_19990606', 'TCGA_CS_5395_19981004', 'TCGA_CS_5396_20010302', 'TCGA_CS_5397_20010315']


In [2]:
# Function to perform global thresholding
def global_threshold(image, threshold_value):
    _, binary = cv2.threshold(image, threshold_value, 255, cv2.THRESH_BINARY)
    return binary

# Function to perform Otsu's thresholding
def otsu_threshold(image):
    _, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary

# Function to perform adaptive thresholding
def adaptive_threshold(image, block_size=11, C=2):
    # block_size must be odd and > 1
    if block_size % 2 == 0:
        block_size += 1
    binary = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, block_size, C)
    return binary

# Preprocessing function: optional resize, Gaussian blur, CLAHE (hist equalization) and normalization
def preprocess_image(image, target_size=None, blur_ksize=5, clahe_clip=2.0, normalize=True, normalize_to_float=False):
    """
    Preprocess a single-channel uint8 image.
    - target_size: (w,h) or None
    - blur_ksize: Gaussian blur kernel size (odd). Set 0 or None to skip blurring.
    - clahe_clip: clip limit for CLAHE. Set 0 or None to skip CLAHE.
    - normalize: if True, apply min-max normalization to the range [0,255] (keeps uint8).
    - normalize_to_float: if True, return float32 array scaled to [0,1] instead of uint8 (overrides normalize range dtype).
    Returns: preprocessed image (uint8 by default, or float32 if normalize_to_float=True)
    """
    proc = image.copy()
    if target_size is not None:
        proc = cv2.resize(proc, target_size, interpolation=cv2.INTER_AREA)
    # apply Gaussian blur if requested
    if blur_ksize and blur_ksize > 0:
        # ensure odd kernel size
        if blur_ksize % 2 == 0:
            blur_ksize += 1
        proc = cv2.GaussianBlur(proc, (blur_ksize, blur_ksize), 0)
    # apply CLAHE
    if clahe_clip and clahe_clip > 0:
        clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
        proc = clahe.apply(proc)

    # Normalization (min-max)
    if normalize:
        # use cv2.normalize to scale to 0-255
        proc = cv2.normalize(proc, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
        # ensure uint8
        proc = proc.astype(np.uint8)

    if normalize_to_float:
        # convert to float32 in [0,1]
        proc = proc.astype(np.float32) / 255.0

    return proc

# Build list of valid image files (walk the tree and filter by extension and filename)
valid_ext = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
image_files = []
excluded_files = []

for root, dirs, files in os.walk(input_dir):
    for fname in files:
        fp = os.path.join(root, fname)
        ext = os.path.splitext(fname)[1].lower()
        # skip obvious mask files and non-image files
        if 'mask' in fname.lower():
            excluded_files.append(fp)
            continue
        if ext in valid_ext:
            image_files.append(fp)
        else:
            excluded_files.append(fp)

print(f"Found {len(image_files)} image files (after excluding masks and non-image files).")
if len(image_files) > 0:
    print("Sample images:")
    for s in image_files[:10]:
        print(' -', s)

if excluded_files:
    print(f"Also found {len(excluded_files)} excluded files (masks / non-images). Showing up to 10:")
    for s in excluded_files[:10]:
        print(' -', s)

if len(image_files) == 0:
    print("No images found. Check the input_dir path and that image files are under it.")

# Limit processing to first 10 images for a quick test
image_files = image_files[:10]
print(f"Processing {len(image_files)} images (first 10).")

# Preprocessing parameters (tweak these if needed)
PREPROCESS_TARGET_SIZE = None  # e.g., (256, 256) or None to keep original
PREPROCESS_BLUR = 5
PREPROCESS_CLAHE_CLIP = 2.0
PREPROCESS_NORMALIZE = True
PREPROCESS_NORMALIZE_TO_FLOAT = False

# Process each image
for idx, img_path in enumerate(image_files, start=1):
    print(f"[{idx}/{len(image_files)}] Reading: {img_path}")
    # Read image
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        print(f"Failed to read image: {img_path}")
        continue
    
    # Preprocess
    pre = preprocess_image(image, target_size=PREPROCESS_TARGET_SIZE, blur_ksize=PREPROCESS_BLUR,
                           clahe_clip=PREPROCESS_CLAHE_CLIP, normalize=PREPROCESS_NORMALIZE,
                           normalize_to_float=PREPROCESS_NORMALIZE_TO_FLOAT)

    # If normalized to float, convert back to uint8 for OpenCV thresholding & saving
    save_pre = pre
    if PREPROCESS_NORMALIZE_TO_FLOAT and isinstance(pre, np.ndarray) and pre.dtype == np.float32:
        save_pre = (pre * 255.0).astype(np.uint8)

    # Save preprocessed (for inspection)
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    try:
        cv2.imwrite(os.path.join(output_dir, f'{base_name}_preproc.png'), save_pre)
    except Exception as e:
        print(f"Warning: could not save preprocessed image for {base_name}: {e}")

    # Apply different thresholding techniques on the preprocessed image
    # Ensure input to thresholding is uint8
    proc_for_thresh = save_pre
    if proc_for_thresh.dtype != np.uint8:
        proc_for_thresh = (proc_for_thresh * 255.0).astype(np.uint8)

    global_binary = global_threshold(proc_for_thresh, 127)  # Fixed threshold at 127
    otsu_binary = otsu_threshold(proc_for_thresh)
    adaptive_binary = adaptive_threshold(proc_for_thresh)
    
    # Create figure for visualization
    plt.figure(figsize=(12, 4))
    
    plt.subplot(141)
    plt.imshow(proc_for_thresh, cmap='gray')
    plt.title('Preprocessed')
    plt.axis('off')
    
    plt.subplot(142)
    plt.imshow(global_binary, cmap='gray')
    plt.title('Global')
    plt.axis('off')
    
    plt.subplot(143)
    plt.imshow(otsu_binary, cmap='gray')
    plt.title("Otsu")
    plt.axis('off')
    
    plt.subplot(144)
    plt.imshow(adaptive_binary, cmap='gray')
    plt.title('Adaptive')
    plt.axis('off')
    
    # Save individual segmentation results (ensure uint8)
    try:
        cv2.imwrite(os.path.join(output_dir, f'{base_name}_global.png'), global_binary.astype(np.uint8))
        cv2.imwrite(os.path.join(output_dir, f'{base_name}_otsu.png'), otsu_binary.astype(np.uint8))
        cv2.imwrite(os.path.join(output_dir, f'{base_name}_adaptive.png'), adaptive_binary.astype(np.uint8))
    except Exception as e:
        print(f"Error saving binary images for {base_name}:", e)

    # Save visualization
    try:
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{base_name}_comparison.png'))
    except Exception as e:
        print(f"Error saving comparison plot for {base_name}:", e)
    finally:
        plt.close()
    
    print(f"Processed and saved: {base_name} -> {output_dir}")


Found 3929 image files (after excluding masks and non-image files).
Sample images:
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_1.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_10.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_11.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_12.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_13.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_14.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_15.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_16.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_17.tif
 - C:/Users/athet/Downloads/archive/kaggle_3m

# Classical Segmentation Methods: A Progressive Approach

1. **Thresholding**
   - Separates image into foreground/background based on pixel intensity
   - Simple but effective for clear intensity differences
   - Methods: Global, Otsu's, and Adaptive (already implemented above)

2. **Canny Edge Detection**
   - Finds boundaries between different regions
   - Good for detecting structural outlines
   - Multi-stage process: noise reduction → gradient → non-maximum suppression → hysteresis

3. **Region Growing**
   - Starts from seed points and grows regions
   - Connects similar pixels into segments
   - Adaptive to local image characteristics

In [3]:
# Additional classical segmentation methods

# Create a new output directory for advanced methods
advanced_output_dir = os.path.abspath('segmentation_advanced')
os.makedirs(advanced_output_dir, exist_ok=True)
print(f"Advanced segmentation results will be saved to: {advanced_output_dir}")

def canny_edge_detection(image, low_threshold=100, high_threshold=200):
    """
    Perform Canny edge detection.
    - image: uint8 input image
    - low_threshold: lower threshold for hysteresis
    - high_threshold: upper threshold for hysteresis
    Returns: binary edge map
    """
    # Ensure uint8 input
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)
    # Apply Canny
    edges = cv2.Canny(image, low_threshold, high_threshold)
    return edges

def region_growing(image, seed_points=None, threshold=10):
    """
    Region growing segmentation.
    - image: input image (uint8)
    - seed_points: list of (x,y) tuples, or None for automatic seed selection
    - threshold: intensity difference threshold for inclusion
    Returns: binary mask of the grown region
    """
    if seed_points is None:
        # Auto-select seeds: use local maxima
        seed_points = []
        blur = cv2.GaussianBlur(image, (5,5), 0)
        for y in range(20, image.shape[0]-20, 40):
            for x in range(20, image.shape[1]-20, 40):
                patch = blur[y-10:y+10, x-10:x+10]
                if np.max(patch) == blur[y,x]:
                    seed_points.append((x,y))
    
    # Initialize mask
    mask = np.zeros_like(image, dtype=np.uint8)
    for seed in seed_points:
        x, y = seed
        if 0 <= y < image.shape[0] and 0 <= x < image.shape[1]:
            mask[y,x] = 255
    
    # Region growing
    seed_intensity = image[mask == 255]
    mean_seed = np.mean(seed_intensity)
    
    while True:
        # Dilate current mask
        dilated = cv2.dilate(mask, np.ones((3,3), np.uint8))
        # Find new pixels
        new_pixels = (dilated - mask) > 0
        # Check intensity similarity
        valid_new = np.abs(image[new_pixels].astype(np.float32) - mean_seed) < threshold
        if not np.any(valid_new):
            break
        # Add valid pixels to mask
        new_mask = np.zeros_like(mask)
        new_mask[new_pixels] = valid_new.astype(np.uint8) * 255
        mask = cv2.bitwise_or(mask, new_mask)
    
    return mask

# Process first 10 images with all methods
image_files = image_files[:10]  # Ensure we're still using just 10 images
print(f"\nApplying additional segmentation methods to {len(image_files)} images...")

for idx, img_path in enumerate(image_files, start=1):
    print(f"[{idx}/{len(image_files)}] Processing: {img_path}")
    
    # Read and preprocess
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        print(f"Failed to read image: {img_path}")
        continue
    
    # Preprocess (using our existing function)
    pre = preprocess_image(image, target_size=PREPROCESS_TARGET_SIZE,
                         blur_ksize=PREPROCESS_BLUR,
                         clahe_clip=PREPROCESS_CLAHE_CLIP,
                         normalize=PREPROCESS_NORMALIZE)
    
    # Apply methods
    edges = canny_edge_detection(pre, 100, 200)
    regions = region_growing(pre)
    
    # Visualize results
    plt.figure(figsize=(15, 3))
    
    plt.subplot(141)
    plt.imshow(pre, cmap='gray')
    plt.title('Preprocessed')
    plt.axis('off')
    
    plt.subplot(142)
    plt.imshow(otsu_threshold(pre), cmap='gray')  # Using previous Otsu for comparison
    plt.title('Otsu (reference)')
    plt.axis('off')
    
    plt.subplot(143)
    plt.imshow(edges, cmap='gray')
    plt.title('Canny Edges')
    plt.axis('off')
    
    plt.subplot(144)
    plt.imshow(regions, cmap='gray')
    plt.title('Region Growing')
    plt.axis('off')
    
    # Save results
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    
    # Save individual results
    try:
        cv2.imwrite(os.path.join(advanced_output_dir, f'{base_name}_edges.png'), edges)
        cv2.imwrite(os.path.join(advanced_output_dir, f'{base_name}_regions.png'), regions)
        # Save comparison figure
        plt.tight_layout()
        plt.savefig(os.path.join(advanced_output_dir, f'{base_name}_all_methods.png'))
    except Exception as e:
        print(f"Error saving results for {base_name}:", e)
    finally:
        plt.close()
    
    print(f"Saved additional results for: {base_name}")

print("\nAll advanced segmentation methods completed. Results saved in:", advanced_output_dir)

# Print summary of output locations
print("\nOutput Directories:")
print(f"1. Basic thresholding results: {output_dir}")
print(f"2. Advanced methods results:   {advanced_output_dir}")

Advanced segmentation results will be saved to: c:\Users\athet\OneDrive\Documents\ULL Admission\machine learning\FInal Project Models\segmentation_advanced

Applying additional segmentation methods to 10 images...
[1/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_1.tif


c:\Users\athet\mycnnenv\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\athet\mycnnenv\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Saved additional results for: TCGA_CS_4941_19960909_1
[2/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_10.tif
Saved additional results for: TCGA_CS_4941_19960909_10
[3/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_11.tif
Saved additional results for: TCGA_CS_4941_19960909_10
[3/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_11.tif
Saved additional results for: TCGA_CS_4941_19960909_11
[4/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_12.tif
Saved additional results for: TCGA_CS_4941_19960909_11
[4/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_12.tif
Saved additional results for: TCGA_CS_4941_19960909_12
[5/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_

KeyboardInterrupt: 

# Clustering-Based Segmentation Methods

1. **K-means Segmentation**
   - Groups pixels into K clusters based on intensity
   - Iteratively refines cluster centers
   - Good for automatic intensity-based segmentation

2. **Watershed Segmentation**
   - Treats image as topographic surface
   - Finds "catchment basins" and "watershed ridge lines"
   - Excellent for separating touching objects

In [ ]:
# Clustering-based segmentation methods

def kmeans_segmentation(image, k=3, attempts=10):
    """
    Perform K-means segmentation on grayscale image.
    - image: uint8 input image
    - k: number of clusters
    - attempts: number of times k-means will run with different initializations
    Returns: labeled image (uint8) where each pixel value represents its cluster
    """
    # Reshape image to a 2D array of pixels and convert to float32
    pixel_values = image.reshape((-1, 1)).astype(np.float32)
    
    # Define criteria and apply kmeans
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(pixel_values, k, None, criteria, attempts,
                                  cv2.KMEANS_RANDOM_CENTERS)
    
    # Convert back to uint8 and original image shape
    centers = centers.astype(np.uint8)
    segmented_image = centers[labels.flatten()].reshape(image.shape)
    
    return segmented_image, labels.reshape(image.shape)

def watershed_segmentation(image):
    """
    Perform watershed segmentation.
    - image: uint8 input image
    Returns: labeled image where each region has a different integer label
    """
    # Noise removal and find sure background
    kernel = np.ones((3,3), np.uint8)
    opening = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel, iterations=2)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # Finding sure foreground
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist_transform, 0.5*dist_transform.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    
    # Finding unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Marker labelling
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    # Apply watershed
    markers = cv2.watershed(cv2.cvtColor(image, cv2.COLOR_GRAY2BGR), markers)
    
    return markers




In [ ]:
# Create a new output directory specifically for clustering results
clustering_output_dir = os.path.abspath('segmentation_clustering')
os.makedirs(clustering_output_dir, exist_ok=True)
print(f"Clustering segmentation results will be saved to: {clustering_output_dir}")

# Process the same 10 images with clustering methods
print(f"\nApplying clustering-based segmentation to {len(image_files)} images...")

for idx, img_path in enumerate(image_files, start=1):
    print(f"[{idx}/{len(image_files)}] Processing: {img_path}")
    
    # Read and preprocess
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        print(f"Failed to read image: {img_path}")
        continue
    
    # Preprocess
    pre = preprocess_image(image, target_size=PREPROCESS_TARGET_SIZE,
                         blur_ksize=PREPROCESS_BLUR,
                         clahe_clip=PREPROCESS_CLAHE_CLIP,
                         normalize=PREPROCESS_NORMALIZE)
    
    # Apply clustering methods
    kmeans_result, kmeans_labels = kmeans_segmentation(pre, k=4)  # 4 clusters for brain tissues
    watershed_result = watershed_segmentation(pre)
    
    # Normalize watershed labels for visualization
    watershed_viz = cv2.normalize(watershed_result.astype(np.float32), None,
                                0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # Create visualization
    plt.figure(figsize=(15, 3))
    
    plt.subplot(141)
    plt.imshow(pre, cmap='gray')
    plt.title('Preprocessed')
    plt.axis('off')
    
    plt.subplot(142)
    plt.imshow(kmeans_result, cmap='nipy_spectral')
    plt.title('K-means (k=4)')
    plt.axis('off')
    
    plt.subplot(143)
    plt.imshow(kmeans_labels, cmap='tab20')
    plt.title('K-means Labels')
    plt.axis('off')
    
    plt.subplot(144)
    plt.imshow(watershed_viz, cmap='nipy_spectral')
    plt.title('Watershed')
    plt.axis('off')
    
    # Save results
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    
    try:
        # Save individual results
        cv2.imwrite(os.path.join(clustering_output_dir, f'{base_name}_kmeans.png'), kmeans_result)
        cv2.imwrite(os.path.join(clustering_output_dir, f'{base_name}_kmeans_labels.png'), 
                   cv2.applyColorMap(kmeans_labels.astype(np.uint8) * 63, cv2.COLORMAP_JET))
        cv2.imwrite(os.path.join(clustering_output_dir, f'{base_name}_watershed.png'), watershed_viz)
        
        # Save comparison figure
        plt.tight_layout()
        plt.savefig(os.path.join(clustering_output_dir, f'{base_name}_clustering_methods.png'))
    except Exception as e:
        print(f"Error saving results for {base_name}:", e)
    finally:
        plt.close()
    
    print(f"Saved clustering results for: {base_name}")

print("\nClustering-based segmentation completed!")
print("\nResults directory structure:")
print(f"1. Basic thresholding:    {output_dir}")
print(f"2. Advanced methods:      {advanced_output_dir}")
print(f"3. Clustering methods:    {clustering_output_dir}")

# Print summary of available results for each image
print("\nFor each processed image, clustering results include:")
print("1. K-means results:")
print(f"   - {clustering_output_dir}/{{name}}_kmeans.png: Intensity-based clustering")
print(f"   - {clustering_output_dir}/{{name}}_kmeans_labels.png: Cluster label visualization")
print("2. Watershed results:")
print(f"   - {clustering_output_dir}/{{name}}_watershed.png: Region-based segmentation")
print("3. Comparison plots:")
print(f"   - {clustering_output_dir}/{{name}}_clustering_methods.png: Side-by-side visualization")

Clustering segmentation results will be saved to: c:\Users\athet\OneDrive\Documents\ULL Admission\machine learning\FInal Project Models\segmentation_clustering

Applying clustering-based segmentation to 10 images...
[1/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_1.tif
Saved clustering results for: TCGA_CS_4941_19960909_1
[2/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_10.tif
Saved clustering results for: TCGA_CS_4941_19960909_10
[3/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_11.tif
Saved clustering results for: TCGA_CS_4941_19960909_11
[4/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_12.tif
Saved clustering results for: TCGA_CS_4941_19960909_12
[5/10] Processing: C:/Users/athet/Downloads/archive/kaggle_3m\TCGA_CS_4941_19960909\TCGA_CS_4941_19960909_13.tif
S

In [ ]:
# Evaluation metrics: Dice, IoU, Accuracy, Precision, Recall
import csv
from collections import OrderedDict

MASK_SUFFIXES = ['_global', '_otsu', '_adaptive', '_edges', '_regions', '_kmeans', '_watershed', '_preproc']
IGNORE_PATTERNS = ['_comparison', '_all_methods', '_clustering_methods', '_kmeans_labels']


def _binarize_mask(arr, thr=127):
    """Convert mask to boolean array based on threshold or non-zero."""
    if arr.dtype != np.uint8:
        # assume float in [0,1]
        arr = (arr * 255.0).astype(np.uint8)
    return (arr > thr).astype(np.uint8)


def compute_confusion(pred, gt):
    """Return TP, FP, FN, TN for binary masks (uint8 0/1 or 0/255)."""
    pred_b = (_binarize_mask(pred) > 0)
    gt_b = (_binarize_mask(gt) > 0)

    tp = int(np.logical_and(pred_b, gt_b).sum())
    fp = int(np.logical_and(pred_b, np.logical_not(gt_b)).sum())
    fn = int(np.logical_and(np.logical_not(pred_b), gt_b).sum())
    tn = int(np.logical_and(np.logical_not(pred_b), np.logical_not(gt_b)).sum())
    return tp, fp, fn, tn


def dice_score(pred, gt):
    tp, fp, fn, tn = compute_confusion(pred, gt)
    if tp == 0 and fp == 0 and fn == 0:
        return 1.0
    denom = (2 * tp + fp + fn)
    return 2 * tp / denom if denom > 0 else 0.0


def iou_score(pred, gt):
    tp, fp, fn, tn = compute_confusion(pred, gt)
    if tp == 0 and fp == 0 and fn == 0:
        return 1.0
    denom = (tp + fp + fn)
    return tp / denom if denom > 0 else 0.0


def accuracy_score(pred, gt):
    tp, fp, fn, tn = compute_confusion(pred, gt)
    total = tp + fp + fn + tn
    return (tp + tn) / total if total > 0 else 0.0


def precision_score(pred, gt):
    tp, fp, fn, tn = compute_confusion(pred, gt)
    denom = (tp + fp)
    if denom == 0:
        return 1.0 if tp == 0 and (tp + fn) == 0 else 0.0
    return tp / denom


def recall_score(pred, gt):
    tp, fp, fn, tn = compute_confusion(pred, gt)
    denom = (tp + fn)
    if denom == 0:
        return 1.0 if tp == 0 and (tp + fp) == 0 else 0.0
    return tp / denom


def evaluate_pair(pred_mask_path, gt_mask_path, thr=127, resize_on_mismatch=False):
    """Compute all metrics for a single prediction/ground-truth pair.
    Returns dict with metrics.
    If resize_on_mismatch=True the pred will be resized to gt shape when needed (with a warning).
    """
    pred = cv2.imread(pred_mask_path, cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
    if pred is None or gt is None:
        raise ValueError(f"Could not read files: {pred_mask_path}, {gt_mask_path}")

    if pred.shape != gt.shape:
        msg = f"Shape mismatch: pred {pred.shape} vs gt {gt.shape} for {os.path.basename(pred_mask_path)}"
        if resize_on_mismatch:
            print("Warning:", msg, "-- resizing prediction to match ground-truth (may distort results).")
            pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_NEAREST)
        else:
            raise ValueError(msg)

    metrics = OrderedDict()
    metrics['dice'] = float(dice_score(pred, gt))
    metrics['iou'] = float(iou_score(pred, gt))
    metrics['accuracy'] = float(accuracy_score(pred, gt))
    metrics['precision'] = float(precision_score(pred, gt))
    metrics['recall'] = float(recall_score(pred, gt))
    return metrics


def find_ground_truth_for(pred_path, search_root=input_dir):
    """Try to find a corresponding ground-truth mask file for a prediction.
    Heuristics:
      - look for files in search_root whose basename contains the prediction base name and 'mask'
      - also accept files named with suffix '_mask' or ending with '_mask.tif/png'
    Returns path or None.
    """
    base = os.path.splitext(os.path.basename(pred_path))[0]
    # strip known prediction suffixes
    base_core = base
    for suf in MASK_SUFFIXES + IGNORE_PATTERNS:
        if base_core.endswith(suf):
            base_core = base_core[: -len(suf)]
            break

    candidates = []
    for root, dirs, files in os.walk(search_root):
        for fname in files:
            low = fname.lower()
            if 'mask' in low and base_core in low:
                candidates.append(os.path.join(root, fname))
    if candidates:
        return candidates[0]

    # fallback pattern: exact base_core + '_mask.*'
    for root, dirs, files in os.walk(search_root):
        for fname in files:
            if fname.lower().startswith(base_core.lower()) and 'mask' in fname.lower():
                return os.path.join(root, fname)

    return None


def _is_figure_like(fname):
    """Return True for filenames that are likely saved figures / comparisons we should skip."""
    low = fname.lower()
    for p in IGNORE_PATTERNS:
        if p in low:
            return True
    # heuristics: very wide images (panels) often have aspect ratio > 2
    try:
        img = cv2.imread(fname, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return False
        h, w = img.shape[:2]
        if w / max(h, 1) > 2.5:
            return True
    except Exception:
        return False
    return False


def evaluate_predictions_in_folder(pred_folder, gt_search_root=input_dir, out_csv=None, verbose=True, resize_on_mismatch=False):
    """Evaluate predicted mask files in pred_folder against ground-truth masks found under gt_search_root.
    This function is more robust: it will prefer prediction files with mask-like suffixes and skip comparison/figure files.
    If resize_on_mismatch=True it will resize prediction masks to ground-truth when shapes differ (not recommended unless necessary).
    """
    results = []
    # list png/jpg files
    all_files = [os.path.join(pred_folder, f) for f in os.listdir(pred_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))]
    all_files.sort()

    # Group by base core name
    grouped = {}
    for fp in all_files:
        name = os.path.splitext(os.path.basename(fp))[0]
        # derive core name by stripping known suffixes
        core = name
        for suf in MASK_SUFFIXES + IGNORE_PATTERNS:
            if core.endswith(suf):
                core = core[: -len(suf)]
                break
        grouped.setdefault(core, []).append(fp)

    for core, files in grouped.items():
        # prefer mask-like predicted files
        pred_candidate = None
        for suf in MASK_SUFFIXES:
            for f in files:
                if f.lower().endswith(suf + '.png') or f.lower().endswith(suf + '.jpg') or f.lower().endswith(suf + '.tif'):
                    pred_candidate = f
                    break
            if pred_candidate:
                break
        # if none found, pick the first non-figure file
        if pred_candidate is None:
            for f in files:
                if not _is_figure_like(f):
                    pred_candidate = f
                    break
        if pred_candidate is None:
            if verbose:
                print(f"No suitable prediction file found for base '{core}' in {pred_folder}; skipping.")
            continue

        gt = find_ground_truth_for(pred_candidate, search_root=gt_search_root)
        if gt is None:
            if verbose:
                print(f"No ground-truth mask found for prediction '{pred_candidate}'; skipping.")
            continue

        try:
            metrics = evaluate_pair(pred_candidate, gt, resize_on_mismatch=resize_on_mismatch)
        except Exception as e:
            if verbose:
                print(f"Error evaluating {pred_candidate} vs {gt}: {e}")
            continue

        row = {'pred': pred_candidate, 'gt': gt}
        row.update(metrics)
        results.append(row)
        if verbose:
            print(f"Evaluated: {os.path.basename(pred_candidate)} | dice={metrics['dice']:.4f} iou={metrics['iou']:.4f}")

    # save CSV
    if out_csv and results:
        keys = list(results[0].keys())
        try:
            with open(out_csv, 'w', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=keys)
                writer.writeheader()
                for r in results:
                    writer.writerow(r)
            if verbose:
                print(f"Saved evaluation CSV to: {out_csv}")
        except Exception as e:
            print(f"Could not save CSV: {e}")

    return results

# Example usage (uncomment and run):
results = evaluate_predictions_in_folder(os.path.abspath('segmentation_results'), out_csv='segmentation_results/eval_basic.csv')
results_adv = evaluate_predictions_in_folder(os.path.abspath('segmentation_advanced'), out_csv='segmentation_advanced/eval_advanced.csv')
results_clust = evaluate_predictions_in_folder(os.path.abspath('segmentation_clustering'), out_csv='segmentation_clustering/eval_clustering.csv')

print("Evaluation metrics functions updated: evaluator now skips comparison figures and prefers actual mask files. Use `evaluate_predictions_in_folder(pred_folder, gt_search_root, resize_on_mismatch=True)` to allow resizing on mismatch (not recommended).")

Evaluated: TCGA_CS_4941_19960909_10_global.png | dice=0.0000 iou=0.0000
Evaluated: TCGA_CS_4941_19960909_11_global.png | dice=0.2231 iou=0.1256
Evaluated: TCGA_CS_4941_19960909_12_global.png | dice=0.3700 iou=0.2270
Evaluated: TCGA_CS_4941_19960909_13_global.png | dice=0.4109 iou=0.2586
Evaluated: TCGA_CS_4941_19960909_14_global.png | dice=0.4494 iou=0.2898
Evaluated: TCGA_CS_4941_19960909_15_global.png | dice=0.3634 iou=0.2220
Evaluated: TCGA_CS_4941_19960909_16_global.png | dice=0.3251 iou=0.1941
Evaluated: TCGA_CS_4941_19960909_17_global.png | dice=0.1540 iou=0.0834
Evaluated: TCGA_CS_4941_19960909_18_global.png | dice=0.0118 iou=0.0059
Evaluated: TCGA_CS_4941_19960909_1_global.png | dice=0.0000 iou=0.0000
Saved evaluation CSV to: segmentation_results/eval_basic.csv
Evaluated: TCGA_CS_4941_19960909_10_edges.png | dice=0.0000 iou=0.0000
Evaluated: TCGA_CS_4941_19960909_11_edges.png | dice=0.0115 iou=0.0058
Evaluated: TCGA_CS_4941_19960909_12_edges.png | dice=0.0380 iou=0.0194
Evaluat

: 